### 4. Create an agent with the MCP tool

The following code creates a project connection in Microsoft Foundry that points to the MCP endpoint of your knowledge base. This connection uses your project managed identity to authenticate to Azure AI Search.

In [1]:
import os 
import requests
from azure.identity import get_bearer_token_provider
from azure.identity import DefaultAzureCredential
from azure.ai.projects.models import PromptAgentDefinition, MCPTool
from azure.ai.projects import AIProjectClient

project_resource_id = os.environ.get("FOUNDRY_PROJECT_RESOURCE_ID", "")
project_connection_name = "azuresearch-connection"
credential = DefaultAzureCredential()
SEARCH_ENDPOINT = os.environ.get("SEARCH_ENDPOINT", "")
base_name = "product-info"

project_endpoint = os.environ.get("FOUNDRY_PROJECT_ENDPOINT", "")
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential)

mcp_endpoint = f"{SEARCH_ENDPOINT}/knowledgebases/{base_name}/mcp?api-version=2025-11-01-Preview"

agent_name = "product-info-agent"
agent_model = os.environ.get("AOAI_GPT_MODEL", "gpt-5.4")


Create a connection between Foundry and Azure AI Search

In [5]:
bearer_token_provider = get_bearer_token_provider(credential, "https://management.azure.com/.default")
headers = {
    "Authorization": f"Bearer {bearer_token_provider()}",
}

response = requests.put(
    f"https://management.azure.com{project_resource_id}/connections/{project_connection_name}?api-version=2025-10-01-preview",
    headers=headers,
    json={
        "name": project_connection_name,
        "type": "Microsoft.MachineLearningServices/workspaces/connections",
        "properties": {
            "authType": "ProjectManagedIdentity",
            "category": "RemoteTool",
            "target": mcp_endpoint,
            "isSharedToAll": True,
            "audience": "https://search.azure.com/",
            "metadata": { "ApiType": "Azure" }
        }
    }
)

response.raise_for_status()
print(f"Connection '{project_connection_name}' created or updated successfully.")

Connection 'azuresearch-connection' created or updated successfully.


Create the Foundry Agent

In [ ]:
instructions = """
You are a helpful assistant that must use the knowledge base to answer all the questions from user. You must never answer from your own knowledge under any circumstances.
Every answer must always provide annotations for using the MCP knowledge base tool and render them as: `【message_idx:search_idx†source_name】`
If you cannot find the answer in the provided knowledge base you must respond with "I don't know".
"""

mcp_kb_tool = MCPTool(
    server_label="knowledge-base",
    server_url=mcp_endpoint,
    require_approval="never",
    allowed_tools=["knowledge_base_retrieve"],
    project_connection_id=project_connection_name
)

agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=agent_model,
        instructions=instructions,
        tools=[mcp_kb_tool]
    )
)

print(f"AI agent '{agent_name}' created or updated successfully")

AI agent 'product-info-agent' created or updated successfully


Chat with the agent and render references:

> Usage: `utils.format_response_with_references(response)` returns a clean answer plus a references list.

> Programmatic usage: `utils.extract_references(response)` returns structured reference objects.

In [2]:
import utils

# Get the OpenAI client for responses and conversations
openai_client = project_client.get_openai_client()
conversation = openai_client.conversations.create()
agent = project_client.agents.get(agent_name)

user_question = "Connect to PostgreSQL from Java"

# Send request that triggers MCP retrieval and citations.
response = openai_client.responses.create(
    conversation=conversation.id,
    tool_choice="required",
    input=user_question,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
)

# Usage 1: render readable answer + references section.
formatted_response = utils.format_response_with_references(response)
print(formatted_response)

Response: To connect to PostgreSQL from Java, use JDBC and put the PostgreSQL JDBC driver JAR on the Java class path.

The JDBC URL format is:
`jdbc:postgresql://host[:port]/[database][parameters]`

Defaults:
- host: `localhost`
- port: `5432`
- database: same as the connecting user

Example JDBC URL:
`jdbc:postgresql://localhost/test?user=fred&password=secret&ssl&sslfactory=org.postgresql.ssl.NonValidatingFactory`

Basic Java example with `DriverManager`:

```java
private static java.sql.Connection connect(String url, String user, String password)
    throws ClassNotFoundException, java.sql.SQLException
{
    Class.forName("org.postgresql.Driver");
    return java.sql.DriverManager.getConnection(url, user, password);
}
```

You can also pass connection settings with `Properties`:

```java
private static java.sql.Connection connect(String url, String user, String password)
    throws ClassNotFoundException, java.sql.SQLException
{
    Class.forName("org.postgresql.Driver");

    java.u

In [3]:
import importlib
import utils

importlib.reload(utils)

if "response" not in globals():
    raise RuntimeError("Run the previous chat cell first to create 'response'.")

# Usage 2: access references as structured objects for downstream processing.
parsed = utils.extract_references(response)

print("Response text:\n")
print(parsed["response_text"])

print("\nStructured references:\n")
for ref in parsed["references"]:
    print(
        {
            "message_idx": ref["message_idx"],
            "search_idx": ref["search_idx"],
            "source_name": ref["source_name"],
            "label": ref["label"],
        }
    )

Response text:

To connect to PostgreSQL from Java, use JDBC and put the PostgreSQL JDBC driver JAR on the Java class path.

The JDBC URL format is:
`jdbc:postgresql://host[:port]/[database][parameters]`

Defaults:
- host: `localhost`
- port: `5432`
- database: same as the connecting user

Example JDBC URL:
`jdbc:postgresql://localhost/test?user=fred&password=secret&ssl&sslfactory=org.postgresql.ssl.NonValidatingFactory`

Basic Java example with `DriverManager`:

```java
private static java.sql.Connection connect(String url, String user, String password)
    throws ClassNotFoundException, java.sql.SQLException
{
    Class.forName("org.postgresql.Driver");
    return java.sql.DriverManager.getConnection(url, user, password);
}
```

You can also pass connection settings with `Properties`:

```java
private static java.sql.Connection connect(String url, String user, String password)
    throws ClassNotFoundException, java.sql.SQLException
{
    Class.forName("org.postgresql.Driver");

    